# Extreme Heat Event — TODO: post title

Config-driven analysis of an extreme heat event. Everything specific to the event —
station, dates, thresholds, warming levels, map area — comes from `event_config.yaml`
in this post folder.

**To run:** from `guidance-website/blog/`, `../relevant-hazard-template/run_event.sh <this post folder>`
(or open this notebook in its post folder and run all). Requires
`guidance-website/relevant-hazard-template/crai_base.ipynb` to have been run once (it builds
the statewide CRAI inputs in `relevant-hazard-template/crai_cache/`).

**Event definition** (`thresholds` in the config): a *hot day / hot night* is a day with
daily max temperature above `tmax_f` **and** daily min temperature at or above `tmin_f`.
A *heatwave event* is a stretch of at least `consecutive_days` such days in a row.

## Setup

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

# event_helpers.py lives in guidance-website/relevant-hazard-template/helpers/.
FRAMEWORK_DIR = os.environ.get("EVENT_FRAMEWORK_DIR", os.path.abspath("../../relevant-hazard-template"))
sys.path.insert(0, os.path.join(FRAMEWORK_DIR, "helpers"))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import event_helpers as eh

cfg = eh.load_config()
if cfg.hazard != "heat":
    raise ValueError(f"This is the heat notebook but the config says hazard: {cfg.hazard}")
cfg.make_output_dirs()

TMAX_THRESH = cfg.thresholds["tmax_f"]
TMIN_THRESH = cfg.thresholds["tmin_f"]
CONSECUTIVE_DAYS = cfg.thresholds.get("consecutive_days", 1)

# Cal-CRAI climate metrics combined with the event metric in Part 5. The CRAI's own
# projected heat-day / warm-night metrics are left out — the event metric replaces them.
EXPOSURE_METRICS = cfg.crai_exposure_metrics or ["median_heat_warning_days"]
LOSS_METRICS = cfg.crai_loss_metrics or [
    "heat_crop_loss_acres",
    "heat_crop_loss_indemnity_amount",
    "avg_age_adjust_heat_hospitalizations_per_10000",
]


def hot_day_hot_night(tmax, tmin, tmax_thresh, tmin_thresh):
    # The one event definition used everywhere below — station, projections, and maps.
    return (tmax > tmax_thresh) & (tmin >= tmin_thresh)

---

## Part 1: What Happened — Observed Station Data

Hourly observations from the **Historical Data Platform (HDP)**, a quality-controlled
archive of weather station records, for `location.station_label`.

In [ ]:
station_id = eh.resolve_hdp_station(cfg.station_label)
station = eh.pull_hdp_station(station_id)

# HDP temperature is in Kelvin, and HDP doesn't support the convert_units processor.
tas_f = (station["tas"] - 273.15) * 9 / 5 + 32
tmax_daily = tas_f.resample(time="D").max()
tmin_daily = tas_f.resample(time="D").min()
print(f"Record: {str(tmax_daily.time.values[0])[:10]} to {str(tmax_daily.time.values[-1])[:10]}")

### The event, by the numbers

Peak temperature and threshold crossings in each configured window (`event.windows`),
then how rare a qualifying day is across the full station record.

In [ ]:
tmax_event_year = tmax_daily.sel(time=tmax_daily.time.dt.year == cfg.event_year)
tmin_event_year = tmin_daily.sel(time=tmin_daily.time.dt.year == cfg.event_year)


def window_label(i, start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    e_str = e.strftime("%-d") if s.month == e.month else e.strftime("%b %-d")
    return f"Heat Wave {i} ({s.strftime('%b %-d')}–{e_str})"


for i, (start, end) in enumerate(cfg.event_windows, start=1):
    tx = tmax_event_year.sel(time=slice(start, end))
    tn = tmin_event_year.sel(time=slice(start, end))
    print(window_label(i, start, end))
    print(f"  Peak Tmax               : {float(tx.max()):.1f}°F")
    print(f"  Days Tmax > {TMAX_THRESH}°F       : {int((tx > TMAX_THRESH).sum())}")
    print(f"  Nights Tmin >= {TMIN_THRESH}°F     : {int((tn >= TMIN_THRESH).sum())}")
    print()

qualifying = hot_day_hot_night(tmax_daily, tmin_daily, TMAX_THRESH, TMIN_THRESH)
n_qualifying, total_days = int(qualifying.sum()), int(qualifying.size)
n_heatwaves = int(eh.count_events(qualifying, "time", CONSECUTIVE_DAYS))
print(f"Days with Tmax > {TMAX_THRESH}°F and Tmin >= {TMIN_THRESH}°F: {n_qualifying} of {total_days} days on record")
print(f"A day meeting both thresholds falls in the {100 - n_qualifying / total_days * 100:.1f}th percentile of all days on record")
print(f"Heatwave events ({CONSECUTIVE_DAYS}+ such days in a row) on record: {n_heatwaves}")

### Daily max/min by year

The event year's daily max/min temperatures against every other year on record, with
the event windows shaded and thresholds marked.

In [ ]:
def daily_frame(da, series):
    df = da.to_pandas().reset_index()
    df.columns = ["date", "temp_f"]
    df["year"] = df["date"].dt.year
    # Anchor every year on a leap year so all years (including leap years) line up.
    df["plot_date"] = pd.to_datetime({"year": 2000, "month": df["date"].dt.month, "day": df["date"].dt.day})
    df["series"] = series
    return df


def anchor_2000(date_str):
    d = pd.Timestamp(date_str)
    return f"2000-{d.month:02d}-{d.day:02d}"


tmax_df = daily_frame(tmax_daily, "Daily Max")
tmin_df = daily_frame(tmin_daily, "Daily Min")
hist_years = sorted(set(tmax_df["year"]) - {cfg.event_year})
# Hoverable years: first year on record + every year divisible by 5
hover_years = {y for y in hist_years if y % 5 == 0} | {hist_years[0]}

lw, alpha = 0.8, 0.3
fig = go.Figure()

for df, color, label in [(tmax_df, "#e0aaaa", "Daily Max"), (tmin_df, "#aac4e0", "Daily Min")]:
    hist = df[df["year"] != cfg.event_year]
    for yr, grp in hist.groupby("year"):
        hoverable = yr in hover_years
        fig.add_trace(go.Scatter(
            x=grp["plot_date"], y=grp["temp_f"], mode="lines",
            line=dict(color=color, width=lw), opacity=alpha, showlegend=False,
            hoverinfo=None if hoverable else "skip",
            hovertemplate=f"{yr} {label}<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>" if hoverable else None,
        ))

for df, color, label in [(tmax_df, "firebrick", "Daily Max"), (tmin_df, "#1a3f7a", "Daily Min")]:
    ev = df[df["year"] == cfg.event_year]
    fig.add_trace(go.Scatter(
        x=ev["plot_date"], y=ev["temp_f"], mode="lines",
        line=dict(color=color, width=lw * 3), name=f"{cfg.event_year} {label}",
        hovertemplate=f"{cfg.event_year} {label}<br>%{{x|%b %d}}<br>%{{y:.1f}} °F<extra></extra>",
    ))

for i, (start, end) in enumerate(cfg.event_windows, start=1):
    x0, x1 = anchor_2000(start), anchor_2000(end)
    fig.add_vrect(x0=x0, x1=x1, fillcolor="darkgoldenrod",
                  opacity=0.12 if i == 1 else 0.20, line_width=0, layer="below")
    fig.add_annotation(x=x0, y=35 - (i - 1) * 4, xanchor="center", showarrow=False,
                       text=window_label(i, start, end).replace(" (", "<br>("),
                       font=dict(size=11, color="darkgoldenrod"), align="center")

fig.add_hline(y=65, line=dict(color="black", width=1.2, dash="dash"),
              annotation_text="65°F AC<br>Threshold", annotation_position="top right",
              annotation_font=dict(size=11, color="black"))
fig.add_hline(y=TMAX_THRESH, line=dict(color="red", width=1.2, dash="dash"),
              annotation_text=f"{TMAX_THRESH}°F Excessive<br>Day Heat", annotation_position="top right",
              annotation_font=dict(size=11, color="red"))
fig.add_hline(y=TMIN_THRESH, line=dict(color="#1a3f7a", width=1.2, dash="dash"),
              annotation_text=f"{TMIN_THRESH}°F Excessive<br>Night Heat", annotation_position="top right",
              annotation_font=dict(size=11, color="#1a3f7a"))

# Legend-only entries (thresholds and windows are labeled on the plot instead)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#e0aaaa", width=lw * 2),
                         name="Historical Daily Max (Every 5 Years)"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#aac4e0", width=lw * 2),
                         name="Historical Daily Min (Every 5 Years)"))

fig.update_layout(
    title=f"{cfg.place_name} ({cfg.station_label}) — Daily Max & Min Temperature by Year",
    xaxis=dict(title="", range=["2000-01-01", "2000-12-31"], showgrid=False, fixedrange=True, tickformat="%b"),
    yaxis=dict(title="Temperature (°F)", showgrid=False, fixedrange=True),
    width=815, height=600,
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"),
    hovermode="closest", dragmode=False,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(l=60, r=90, t=50, b=50),
)
fig.show(config=eh.PLOTLY_CONFIG)
eh.save_figure(fig, cfg, "daily_temp_by_year")

---

## Part 2: Historical Context — How Unusual Was This?

The event year's annual mean temperature anomaly relative to the full-record mean,
with a 5-year rolling mean. Years after the event are left out.

In [ ]:
annual_mean = tas_f.resample(time="D").mean().resample(time="YE").mean()
anomaly = annual_mean - float(tas_f.mean())
anomaly = anomaly.sel(time=anomaly.time.dt.year <= cfg.event_year)

fig = eh.plot_annual_bars(
    anomaly.time.dt.year.values, anomaly.values, cfg.event_year,
    title=f"Annual Mean Temperature Anomaly relative to full-record mean at {cfg.station_label}",
    yaxis_title="Temperature anomaly (°F)", unit="°F",
)
eh.save_figure(fig, cfg, "annual_anomaly")

---

## Part 3: How Frequently Will This Occur in the Future?

Dynamically downscaled **WRF** projections (UCLA, 3 km, SSP3-7.0) for `location.clip`
at each warming level in `projection.warming_levels`.

**Methodology:** extreme heat is defined relative to each simulation's own local climate:

1. **Threshold calibration at the baseline warming level** — for each simulation and
   grid cell, the `projection.percentile` percentile of daily max and of daily min
   temperature at `projection.baseline_warming_level`.
2. **Co-occurrence** — at each warming level, days where daily max exceeds the max
   threshold **and** daily min reaches the min threshold.
3. **Aggregation** — days per year, averaged across grid cells and then simulations,
   converted to a return period (1-in-X years).

In [ ]:
# Pulled once and reused by Part 4 — simulations in projection.exclude_simulations are dropped.
t2max = eh.pull_wrf_warming_levels("t2max", cfg, "degF")
t2min = eh.pull_wrf_warming_levels("t2min", cfg, "degF")
print(dict(t2max.sizes))
print("simulations:", list(t2max.sim.values))

In [ ]:
# Per-simulation, per-cell thresholds at the baseline warming level -> (sim, y, x),
# broadcast across every warming level.
tmax_threshold = (t2max.sel(warming_level=cfg.baseline_warming_level)
                  .quantile(cfg.percentile, dim="time_delta").drop_vars("quantile"))
tmin_threshold = (t2min.sel(warming_level=cfg.baseline_warming_level)
                  .quantile(cfg.percentile, dim="time_delta").drop_vars("quantile"))

events = hot_day_hot_night(t2max, t2min, tmax_threshold, tmin_threshold)
events_per_year = events.sum("time_delta").mean(["x", "y"]).mean("sim") / cfg.window_years

fig, return_periods = eh.plot_return_periods(
    events_per_year.values, cfg, title="How Much More Frequent Will Days Like This Be?"
)
eh.save_figure(fig, cfg, "return_period_by_gwl")
for wl, rate, rp in zip(cfg.warming_levels, events_per_year.values, return_periods):
    print(f"{wl}°C: {rate:.3f} days/yr -> 1-in-{rp:.1f} yrs")

---

## Part 4: Where — Tract-Level Event Metrics

The same WRF data, now with the **fixed** thresholds from the config and without
averaging over space, mapped onto the census tracts inside `location.zip_shapefile`:

- **Hot days / hot nights** — days per year with Tmax > `tmax_f` and Tmin ≥ `tmin_f`
- **Heatwave events** — events per year with at least `consecutive_days` such days in a row

Each tract takes its nearest WRF grid cell. Values are averaged across simulations,
written to `crai_layers/tract_metrics/`, and min-max standardized within the area for Part 5.

In [ ]:
fixed = hot_day_hot_night(t2max, t2min, TMAX_THRESH, TMIN_THRESH)
gridded = {
    "hot_days_hot_nights": fixed.sum("time_delta").mean("sim") / cfg.window_years,
    "heatwave_events": eh.count_events(fixed, "time_delta", CONSECUTIVE_DAYS).mean("sim") / cfg.window_years,
}

zip_codes = eh.load_zip_codes(cfg.zip_shapefile)
tracts = eh.tracts_in_area(cfg, zip_codes)
zip_labels = eh.geoid_to_zip_label(tracts, zip_codes)

tract_layers = {metric: eh.tract_metric_layers(da, metric, tracts, cfg) for metric, da in gridded.items()}

In [ ]:
eh.tabbed_choropleth(
    tract_layers["hot_days_hot_nights"], "hot_days_hot_nights",
    f"{TMAX_THRESH}°F Day / {TMIN_THRESH}°F Night — Days per Year",
    "hot_days_hot_nights_by_gwl", cfg, tracts, zip_labels, zip_codes,
    hover_label="Days per year", colorbar_title="Days / year",
)
eh.tabbed_choropleth(
    tract_layers["heatwave_events"], "heatwave_events",
    f"Heatwaves ({CONSECUTIVE_DAYS}+ days in a row) — Events per Year",
    "heatwave_events_by_gwl", cfg, tracts, zip_labels, zip_codes,
    hover_label="Events per year", colorbar_title="Events / year",
);

---

## Part 5: Who — Climate Risk and Adaptation Index (CRAI)

For each event metric and warming level:

1. **Hazard score** — the event metric joins the Cal-CRAI heat exposure and loss metrics
   (`crai.exposure_metrics` / `crai.loss_metrics`, defaults set in Setup), combined as
   exposure × loss and standardized statewide.
2. **CRAI score** — adaptive capacity (built + governance + natural + society domain
   scores, from `crai_cache/`) divided by the hazard score.

Written to `crai_layers/hazard_scores/` and `crai_layers/crai_scores/`. The maps show the
change in hazard score at each warming level relative to the baseline.

In [ ]:
adaptive_capacity, climate_metrics = eh.load_crai_cache(cfg)

hazard_layers, crai_layers = {}, {}
for metric, layers in tract_layers.items():
    hazard_layers[metric], crai_layers[metric] = {}, {}
    for gwl in cfg.gwl_order:
        hazard = eh.hazard_scores(layers[gwl], metric, climate_metrics, EXPOSURE_METRICS, LOSS_METRICS)
        crai = eh.crai_scores(hazard, adaptive_capacity)
        hazard.to_csv(cfg.hazard_score_csv(metric, gwl), index=False)
        crai.to_csv(cfg.crai_score_csv(metric, gwl), index=False)
        hazard_layers[metric][gwl] = tracts.merge(hazard, on="GEOID")
        crai_layers[metric][gwl] = tracts.merge(crai, on="GEOID")
    print(metric, {g: round(float(crai_layers[metric][g]["crai_score"].median()), 3) for g in cfg.gwl_order},
          "(median CRAI score in area)")

In [ ]:
for metric, title in [
    ("hot_days_hot_nights", "Hot Days and Warm Nights — Difference from Baseline"),
    ("heatwave_events", "Heatwaves — Difference from Baseline"),
]:
    eh.tabbed_delta_choropleth(
        eh.deltas_vs_baseline(hazard_layers[metric], "hazard_score", cfg.gwl_order),
        title, f"{metric}_delta_by_gwl", cfg, zip_labels, zip_codes,
    )

---

## Part 6: So What?

TODO: what the numbers above mean for this place — who is most exposed, and how that
changes with warming.

## Part 7: What's Next?

TODO